# N058 · Top-K、频率与流式维护

**目标：** 用大小K的堆保留当前所需候选。

**先修：** N057, N020, N041。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 小根堆维护最大K项；频率Top-K；流式输入；K边界。

## 从问题到算法

维护最大的k个值时，最需要知道的是这k个候选中最小的那个，因此使用大小k的最小堆。新值不大于堆顶就不可能挤进前k。高频元素先计数再选k个键，不能把相同频次误认为答案唯一。本章为稳定展示，频次并列时取数值较小者。

## 最小实现

**本章接口：** `kth_largest_heap(nums, k)`、`top_k_frequent(nums, k)`、`KthLargest`

In [1]:
import heapq
from collections import Counter

def kth_largest_heap(nums,k):
    if not 1<=k<=len(nums): raise ValueError('invalid rank')
    heap=[]
    for x in nums:
        if len(heap)<k: heapq.heappush(heap,x)
        elif x>heap[0]: heapq.heapreplace(heap,x)
    return heap[0]

def top_k_frequent(nums,k):
    counts=Counter(nums)
    if not 0<=k<=len(counts): raise ValueError('invalid number of distinct keys')
    return heapq.nlargest(k,counts,key=lambda x:(counts[x],-x))

class KthLargest:
    def __init__(self,k,nums):
        if k<1: raise ValueError('positive k required')
        self.k=k; self.heap=[]
        for x in nums: self.add(x)
    def add(self,value):
        heapq.heappush(self.heap,value)
        if len(self.heap)>self.k: heapq.heappop(self.heap)
        return self.heap[0]

## 正确性、前提与复杂度

对输入前缀归纳：堆保存其最大的min(k,t)个元素；插入后删除最小者仍保留该集合。流长度不足k时本章返回当前最小值，并不称它为已存在的“第k大”。

**代价：** Top-k扫描O(n log k)、O(k)空间；频次版O(n+u log k)、O(u)空间。

## 小实例与可视化

先手算，再运行。下表由当前内核中的代码和显式小实例生成；不是预制答案图片。

In [2]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

stream=KthLargest(3,[4,5,8,2]); rows=[]
for x in [3,5,10,9,4]:
    result=stream.add(x); rows.append((x,sorted(stream.heap),result))
show_table(['新值','保留的Top3','第3大'],rows)

新值,保留的Top3,第3大
3,"[4, 5, 8]",4
5,"[5, 5, 8]",5
10,"[5, 8, 10]",5
9,"[8, 9, 10]",8
4,"[8, 9, 10]",8


## 自动测试

以下断言检查实现契约。测试通过不等同于一般性证明，也不表示已经提交 LeetCode 官方评测。

In [3]:
assert kth_largest_heap([3,2,1,5,6,4],2)==5
assert set(top_k_frequent([1,1,1,2,2,3],2))=={1,2}
stream=KthLargest(3,[4,5,8,2]); assert [stream.add(x) for x in [3,5,10,9,4]]==[4,5,5,8,8]
from random import Random
rng=Random(58)
for k in range(1,8):
    seen=[]; stream=KthLargest(k,[])
    for _ in range(40):
        x=rng.randrange(20); seen.append(x)
        answer=stream.add(x)
        assert answer==sorted(seen)[-min(k,len(seen))]
print('N058: 所有本章断言通过')

N058: 所有本章断言通过


## 练习：先独立完成

**练习 1：** 比较排序、快选与堆的时间空间。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

**练习 2：** 定义频率并列时的合法输出判定。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

未完成练习不阻断教学区 Run All。验收时解释为什么正确，不只展示运行截图。

## 代表题与迁移

- 215. Kth Largest Element in an Array（canonical）
- 347. Top K Frequent Elements（canonical）
- 703. Kth Largest Element in a Stream（canonical）

这些题用于知识映射；本章实现遵循上文明确的教学契约。除原规格注明已核对身份的条目外，本轮未重新联网核验全部题面、收费权限或平台语言支持。不要把同概念教学实例当作所有官方题的完整答案。

## 自测与下一步

不看代码，复述状态、不变量和一个失效条件；改变一个输入约束，判断实现是否仍成立。确认能解释边界测试后，按 `specs/curriculum.json` 的 `learning_order` 进入下一章。